In [6]:
import time, numpy as np
from multiprocessing import Pool, cpu_count

# worker functions go in a file so multiprocessing works inside Jupyter
open("calc_mod.py", "w").write('''
def slow(t):   # unoptimized: plain Python loop
    op, a, b = t
    if op == "add":      return [x + y for x, y in zip(a, b)]
    if op == "subtract": return [x - y for x, y in zip(a, b)]
    if op == "multiply": return [x * y for x, y in zip(a, b)]
    if op == "divide":   return [x / y for x, y in zip(a, b)]

def fast(t):   # optimized: NumPy
    op, a, b = t
    if op == "add":      return a + b
    if op == "subtract": return a - b
    if op == "multiply": return a * b
    if op == "divide":   return a / b
''')
from calc_mod import slow, fast

N = 1_000_000
W = cpu_count()
ops = ["add", "subtract", "multiply", "divide"]
a = np.random.rand(N) * 100
b = np.random.rand(N) * 100 + 1          # +1 avoids divide by zero
al, bl = a.tolist(), b.tolist()

def serial(func, x, y):
    for op in ops: func((op, x, y))

def parallel(func, x, y, pool):
    xs, ys = np.array_split(x, W), np.array_split(y, W)   # one chunk per core
    for op in ops: pool.map(func, [(op, p, q) for p, q in zip(xs, ys)])

def timeit(f, *args):
    t = time.perf_counter(); f(*args); return time.perf_counter() - t

with Pool(W) as pool:
    pool.map(abs, range(W))              # warm-up so pool startup isn't timed
    results = {
        "Serial   unoptimized": timeit(serial, slow, al, bl),
        "Serial   optimized":   timeit(serial, fast, a, b),
        "Parallel unoptimized": timeit(parallel, slow, al, bl, pool),
        "Parallel optimized":   timeit(parallel, fast, a, b, pool),
    }

print(f"N = {N}, workers = {W}")
for k, v in results.items():
    print(f"{k}: {v:.4f} s")

N = 1000000, workers = 8
Serial   unoptimized: 0.7953 s
Serial   optimized: 0.0206 s
Parallel unoptimized: 6.6065 s
Parallel optimized: 0.2174 s
